## choice

In [ ]:
VERSION = "1.1.1"
PERCENTAGE = 100
STYLE = "pisco_euclid"
SPLIT = "dataset"

## TRAIN DATASETS FOR NN
# "1.*.*": "TRAINING DATASETS => g round(3) + 10 gal/image + g<=|0.06| + e gauss + seeds 22 + airy + big_psf",  # noqa
# "1.0.1": "25_000",
# "1.0.2": "25_000 + lhs_pos",
# "1.0.3": "25_000 + pixel_out",
# "1.0.4": "25_000 with 1 gal/image",
# "1.0.5": "25_000 + 25'%' parametric galaxies",
# "1.0.6": "25_000 + 25'%' real galaxies",
# "1.1.1": "50_000",
# "1.1.2": "50_000 + lhs_pos",
# "1.1.3": "50_000 + pixel_out",
# "1.1.4": "50_000 with 1 gal/image",
# "1.1.5": "50_000 + 25'%' parametric galaxies",
# "1.1.6": "50_000 + 25'%' real galaxies",

## TEST DATASETS FOR NN & S++
# "2.*.*": "TEST DATASETS => g round(3) + 10 gal/patch + g<=|0.06| + e gauss + seeds 22 + airy + small_psf",  # noqa
# "2.0.1": "1_000",
# "2.0.2": "2_000 (1000 x 2 : with shape noise cancellation with add noise)",
# "2.0.3": "4_000 (1000 x 4 : with shape noise cancellation with add noise + sub noise)",
# "2.0.4": "1_000 + lhs_pos",
# "2.0.5": "1_000 + pixel_out",
# "2.0.6": "1_000 with 1 gal/image",
# "2.0.7": "1_000 + 25'%' parametric galaxies",
# "2.0.8": "1_000 + 25'%' real galaxies",
# "2.1.1": "25_000",
# "2.1.2": "50_000 (25_000 x 2 : with shape noise cancellation with add noise)",
# "2.1.3": "100_000 (25_000 x 4 : with shape noise cancellation with add noise + sub noise)",
# "2.1.4": "25_000 + lhs_pos",
# "2.1.5": "25_000 + pixel out",
# "2.1.6": "25_000 with 1 gal/image",
# "2.1.7": "25_000 + 25'%' parametric galaxies",
# "2.1.8": "25_000 + 25'%' real galaxies",
# "2.2.0": "100_000",

## libs

In [ ]:
import resource

import galsim
import hiplot
import holoviews
import numpy
import panel
import tensorflow_datasets as tfds
from astropy.stats import mad_std
from scipy import stats
from pandas import DataFrame, concat, read_csv
from functions_filters import get_field_cosmos
from functions_images import GSPARAMS, build_psf, get_constants
from functions_lhs import adding_cosmos_catalog_galaxies, draw_lhs, draw_lhs_gal, project_lhs_gal
from functions_redshift import fit_g_vs_z, generate_shear_scaled, redshift_vs_shear_data
from functions_cdfs import (
    compute_centers_and_edges,
    compute_which_bin,
    get_projected,
    get_projected_magnitude,
    load_data_for_inverse_functions,
)
from plots import (
    before_after,
    check_image,
    check_psf,
    combined_plot,
    comparaison_3d,
    diff_binned_continuous,
    diff_real_param_cosmos,
    heatmaps_inputs,
    histo_galaxies,
    histo_images,
    images_psf,
    interactive_dist,
    interactive_hist,
    interactive_pairplot,
    large_ellps,
    lhs_visu,
    lhs_visu_single,
    noise_shape,
    plot_example,
    sky_coverage,
    style_norma,
    visu_inputs_ouputs,
)
from utils import (
    DIR_DATASETS,
    DIRS,
    get_indices_snc,
    get_pairs,
    getting_norms,
    save_psf,
    scale,
    slice_csv,
    binned_style,
)

holoviews.extension("bokeh", "plotly")
panel.extension()

resource.setrlimit(resource.RLIMIT_NOFILE, (50000, 50000))

DIR_VERSION = DIRS["pisco_euclid"] / VERSION
DIR_SE = DIRS["pisco_euclid_se"] / VERSION
DIR_PLOTS = DIRS["pisco_euclid_plots"] / VERSION
DIR_PLOTS.mkdir(parents=True, exist_ok=True)

## load all params

In [ ]:
dataset, ds_info = tfds.load(
    f"{STYLE}:{VERSION}",
    split=f"{SPLIT}[:{PERCENTAGE}%]",
    data_dir=DIR_DATASETS,
    with_info=True,
    shuffle_files=False,  # no True to match the csv files
)
print(ds_info.metadata)
print(dataset.element_spec.keys())

TOTAL_ELEMENTS = dataset.cardinality().numpy()

hypercube_img = slice_csv(DIR_VERSION / "images_parameters_true.csv", TOTAL_ELEMENTS)
hypercube_gal = slice_csv(DIR_VERSION / "galaxies_parameters_true.csv", TOTAL_ELEMENTS * 10)

keys_dataset = ["image"]
all_params_dataset = dataset.map(lambda x: {key: x[key] for key in keys_dataset})
all_params = list(all_params_dataset.as_numpy_iterator())
images = numpy.stack([item["image"] for item in all_params])

del all_params_dataset, all_params, dataset

# compute stats  (15_000 images max)

In [ ]:
if not (DIR_VERSION / "all_statistics.csv").exists():
    DataFrame(
        [
            {
                "min_images": numpy.amin(images),
                "max_images": numpy.amax(images),
                "mean_images": numpy.mean(images),
                "std_images": mad_std(images),
                "min_avg_g": numpy.amin(hypercube_img[["avg_g1", "avg_g2"]].to_numpy()),
                "max_avg_g": numpy.amax(hypercube_img[["avg_g1", "avg_g2"]].to_numpy()),
                "min_avg_obs_e": numpy.amin(hypercube_img[["avg_obs_e1", "avg_obs_e2"]].to_numpy()),
                "max_avg_obs_e": numpy.amax(hypercube_img[["avg_obs_e1", "avg_obs_e2"]].to_numpy()),
                "min_avg_int_e": numpy.amin(hypercube_img[["avg_int_e1", "avg_int_e2"]].to_numpy()),
                "max_avg_int_e": numpy.amax(hypercube_img[["avg_int_e1", "avg_int_e2"]].to_numpy()),
                "min_obs_e": numpy.amin(hypercube_gal[["obs_e1", "obs_e2"]].to_numpy()),
                "max_obs_e": numpy.amax(hypercube_gal[["obs_e1", "obs_e2"]].to_numpy()),
                "min_int_e": numpy.amin(hypercube_gal[["int_e1", "int_e2"]].to_numpy()),
                "max_int_e": numpy.amax(hypercube_gal[["int_e1", "int_e2"]].to_numpy()),
            }
        ]
    ).to_csv(DIR_VERSION / "all_statistics.csv", index=False)

# shape noise cancellation

In [ ]:
grouped_values, duplicate_pairs_image = get_indices_snc(hypercube_gal)

In [ ]:
CHOICE = 5
apply_arcsinh = True

if len(duplicate_pairs_image) != 0:
    indices = duplicate_pairs_image[CHOICE]

    noise_shape(DIR_PLOTS, images[numpy.array(indices)], hypercube_img.iloc[indices], apply_arcsinh)
    del indices

del apply_arcsinh, CHOICE

# histo hypercubes

In [ ]:
hypercube_img["avg_obs_e1"].min(), hypercube_img["avg_obs_e1"].max()

In [ ]:
hypercube_img["avg_int_e1"].min(), hypercube_img["avg_int_e1"].max()

In [ ]:
hypercube_img["avg_int_e2"].min(), hypercube_img["avg_int_e2"].max()

In [ ]:
histo_images(hypercube_img, DIR_PLOTS)

In [ ]:
histo_galaxies(hypercube_gal, dir_plots=DIR_PLOTS)

# params

In [ ]:
pairs_img = [("avg_g1", "avg_g2"), ("avg_obs_e1", "avg_obs_e2"), ("avg_int_e1", "avg_int_e2")]
pairs_gal = [
    ("int_e1", "int_e2"),
    ("obs_e1", "obs_e2"),
    ("x", "y"),
    ("mag", "hlr"),
    ("mag", "sersic"),
]
all_pairs = get_pairs(pairs_img, hypercube_img, pairs_gal, hypercube_gal)

In [ ]:
alpha = 0.3
size = 2
lhs_visu(DIR_PLOTS, all_pairs, alpha, size)

In [ ]:
NB = 7
alpha = 0.5

lhs_visu_single(DIR_PLOTS, all_pairs[NB], alpha)

# pairplots

In [ ]:
size_dots = 2
columns_drop_img = ["seeds", "hashed_key", "k"]
interactive_pairplot(
    hypercube_img.drop(columns=columns_drop_img, inplace=False),
    size_dots,
    DIR_PLOTS / "pairplot_images",
    width=1250,
    height=1250,
)

In [ ]:
size_dots = 2
columns_drop_gal = ["int_e1", "int_e2"]
interactive_pairplot(
    hypercube_gal.drop(columns=columns_drop_gal, inplace=False),
    size_dots,
    DIR_PLOTS / "pairplot_gal_params",
    width=1350,
    height=1350,
)

# examples

In [ ]:
print(selected_number := numpy.random.permutation(images.shape[0])[:3])  # numpy.array([6, 8])

selected_images = images[selected_number]
plot_example(selected_images, DIR_PLOTS)
del selected_number, selected_images

In [ ]:
visu_inputs_ouputs(dbb={"inputs": hypercube_img, "simu": images}, loggy=True)

# large single ellipticities

In [ ]:
WHICH = "obs"
# WHICH = "int"
THRESH_1 = 0.68
THRESH_2 = 0.02

conditions = (abs(hypercube_gal[f"{WHICH}_e1"]) > THRESH_1) & (
    abs(hypercube_gal[f"{WHICH}_e2"]) < THRESH_2
)

filtered_inputs = hypercube_gal[conditions]
print("cases possible", index_numbers := filtered_inputs.index.tolist())

if len(index_numbers) != 0:
    select_number_e = numpy.random.permutation(numpy.array(index_numbers))[:3]
    large_ellps(
        selected_images=images[select_number_e // 10],
        elps1=hypercube_gal[f"{WHICH}_e1"][select_number_e],
        elps2=hypercube_gal[f"{WHICH}_e2"][select_number_e],
        number=select_number_e // 10,
        directory=DIR_PLOTS,
        name=f"single_{WHICH}",
    )

    del THRESH_1, THRESH_2, WHICH, conditions, filtered_inputs, select_number_e

# norms outputs

In [ ]:
images_norms = getting_norms(data=images)
panel.Tabs(
    ("HISTO", interactive_hist(data=images_norms)), ("DIST", interactive_dist(data=images_norms))
)

# parallel plots

In [ ]:
columns_drop = ["seeds", "hashed_key", "k"]

_ = hiplot.Experiment.from_dataframe(
    dataframe=concat(
        (images_norms, hypercube_img.drop(columns=columns_drop, inplace=False)), axis=1
    )
).display(force_full_width=True)

# heatmaps

In [ ]:
columns_drop_img = ["seeds", "hashed_key", "k"]
columns_drop_gal = ["int_e1", "int_e2"]

heatmaps_inputs(
    hypercube_gal.drop(columns=columns_drop_gal, inplace=False),
    concat((images_norms, hypercube_img.drop(columns=columns_drop_img, inplace=False)), axis=1),
    DIR_PLOTS,
)

# scaling

In [ ]:
sigma = read_csv(DIR_VERSION / "all_statistics.csv")["std_images"][0]

style_norma(images.flatten(), style="regular", directory=DIR_PLOTS)

In [ ]:
style_norma(
    numpy.arcsinh(images.flatten() / (sigma * 3)), style="arcsinh-sig-3", directory=DIR_PLOTS
)

In [ ]:
NUM = 0

before_after(images[NUM, :, :], DIR_PLOTS)
del NUM

# psf sextractor

In [ ]:
all_psf = list(DIRS["psf_se"].glob("*.fits"))

for each_psf in all_psf:
    print(each_psf)

CHOICE = 0
print("CHOICE", all_psf[CHOICE])
check_psf(all_psf[CHOICE])

In [ ]:
style = "airy"  #  airy / euclid
psf = build_psf(style)
image_size = 81  # must have odd size for S++
pixel_scale = 0.06

save_psf(DIRS["psf"], style, psf=psf, image_size=image_size, pixel_scale=pixel_scale)
del style, psf, image_size, pixel_scale

# check assoc vs image (if sextractor database created)


In [ ]:
CHOICE = "000110"
compress = True

check_image(DIR_SE / f"{CHOICE}_simu", DIR_PLOTS, compress)
del CHOICE, compress

# test real vs parametric cosmos catalog

In [ ]:
CHOICE = 0
SAMPLE = "23.5"
PIXEL_SCALE, PATCH_SIZE, _, EXPTIME, _, _, _, _ = get_constants()

test_cosmos_cat = galsim.COSMOSCatalog(
    sample=SAMPLE,
    dir=DIRS["cosmos"],
    use_real=True,
    exclusion_level="none",
    exptime=EXPTIME,
)

common_kwargs = dict(index=CHOICE, noise_pad_size=0, deep=False, sersic_prec=0.0, gsparams=GSPARAMS)
real_gal = test_cosmos_cat.makeGalaxy(gal_type="real", **common_kwargs)
param_gal = test_cosmos_cat.makeGalaxy(gal_type="parametric", **common_kwargs)


diff_real_param_cosmos(CHOICE, PIXEL_SCALE, PATCH_SIZE, real_gal, param_gal, DIRS["cosmos_plots"])

# redshift vs shear


In [ ]:
N_SAMPELS = 1000
hypercube_redshift = draw_lhs(22222, N_SAMPELS, dim=2)
hypercube_redshift = scale(array=hypercube_redshift, low=-0.055, high=0.055)

z_grid = numpy.linspace(0.2, 5, 10000)
z_pdf = z_grid**2 * numpy.exp(-z_grid / 0.8)

z_data, g_data = redshift_vs_shear_data()

deg = 1
shunsheng_fn = fit_g_vs_z(g_data, z_data, deg=deg)

g_magnitude = numpy.sqrt(hypercube_redshift[:, 0] ** 2 + hypercube_redshift[:, 1] ** 2)
z, g1_scaled, g2_scaled = generate_shear_scaled(
    hypercube_redshift[:, 0], hypercube_redshift[:, 1], shunsheng_fn, z_grid, z_pdf, g_magnitude
)

combined_plot(
    DIRS["wip"],
    z,
    z_grid,
    z_pdf,
    z_data,
    g_data,
    shunsheng_fn,
    hypercube_redshift,
    g1_scaled,
    g2_scaled,
)

# mollmoll


In [ ]:
fields_23 = get_field_cosmos(sample="23.5")
fields_25 = get_field_cosmos(sample="25.2")
ra_23 = numpy.array(fields_23["RA"])
dec_23 = numpy.array(fields_23["DEC"])
ra_25 = numpy.array(fields_25["RA"])
dec_25 = numpy.array(fields_25["DEC"])

sky_coverage(ra_23, dec_23, ra_25, dec_25, DIRS["cosmos_plots"])

# psf

In [ ]:
VERSION_PSF = "0.0.6"
PERCENTAGE_PSF = 100
STYLE_PSF = "psf"
SPLIT_PSF = "dataset"

In [ ]:
dataset, ds_info = tfds.load(
    f"{STYLE_PSF}:{VERSION_PSF}",
    split=f"{SPLIT_PSF}[:{PERCENTAGE_PSF}%]",
    data_dir=DIR_DATASETS,
    with_info=True,
)
print(ds_info.metadata)
print(dataset.element_spec.keys())

keys_dataset = ["image_psf"]
all_params_dataset = dataset.map(lambda x: {key: x[key] for key in keys_dataset})
all_params = list(all_params_dataset.as_numpy_iterator())
image_psf = numpy.stack([item["image_psf"] for item in all_params])

del all_params_dataset, all_params, dataset

In [ ]:
CHOICE = 0
images_psf(DIRS["psf_plots"] / VERSION_PSF, image_psf[CHOICE])

# check replacing


In [ ]:
NB_OF_GALAXIES = 2_000
PERCENTAGE_COSMOS = 10
version = "9999"
seed_database = 2222

hypercube = draw_lhs_gal(False, seed_database, NB_OF_GALAXIES)
hypercube = project_lhs_gal(hypercube=hypercube, ellips_style="gaussian", pixel_out=False)

In [ ]:
nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_GALAXIES / 100)
rng_local = numpy.random.default_rng(seed=seed_database)
indices = rng_local.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)
indices = numpy.random.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)

check_columns = [0, 1, 4, 5, 6]  # e1 e2 mag hlr sersic
catalog_targets = hypercube[:, check_columns][indices]

filtered_params = read_csv(DIRS["cosmos"] / "filtered_params.csv")
check_columns_cosmos = ["e1", "e2", "mag", "hlr", "sersic"]

catalog_features = filtered_params[check_columns_cosmos].values

results_features = {}
for SCALING in ["min_max", "z_score"]:
    for NORMA in ["euclidian", "mahalanobis"]:
        lhs_gal = adding_cosmos_catalog_galaxies(hypercube, indices, version, SCALING, NORMA)
        results_features[(SCALING, NORMA)] = lhs_gal[numpy.ix_(indices, check_columns)]

In [ ]:
with_all_catalog = False
NB_CHOICES = 10
# numpy.random.seed(22)
choices = numpy.random.choice(nb_galaxies_cosmos, size=NB_CHOICES, replace=False)
choices

comparaison_3d(
    catalog_features, catalog_targets, results_features, choices, DIRS["wip"], with_all_catalog
)